In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

In [2]:
# 1. Toy sentences
sentences = [
    "steve jobs founded apple in california",
    "elon musk leads tesla and spacex",
    "tim cook visited apple headquarters yesterday",
    "satya nadella manages microsoft successfully",
    "google hired an engineer from apple",
    "jeff bezos stepped down from amazon"
]

# Labels:
# 0 = PAD (reserved for mask)
# 1 = O (Regular word / Other)
# 2 = PER (Person name)
# 3 = ORG (Organization / Company)

labels = [
    [2, 2, 1, 3, 1, 1],       # steve(PER) jobs(PER) founded(O) apple(ORG) in(O) california(O)
    [2, 2, 1, 3, 1, 3],       # elon(PER) musk(PER) leads(O) tesla(ORG) and(O) spacex(ORG)
    [2, 2, 1, 3, 1, 1],       # tim(PER) cook(PER) visited(O) apple(ORG) headquarters(O) yesterday(O)
    [2, 2, 1, 3, 1],          # satya(PER) nadella(PER) manages(O) microsoft(ORG) successfully(O)
    [3, 1, 1, 1, 1, 3],       # google(ORG) hired(O) an(O) engineer(O) from(O) apple(ORG)
    [2, 2, 1, 1, 1, 3]        # jeff(PER) bezos(PER) stepped(O) down(O) from(O) amazon(ORG)
]

In [4]:
max_len = 6
vectorizer = tf.keras.layers.TextVectorization(output_mode='int',output_sequence_length=max_len)

vectorizer.adapt(sentences)

In [6]:
vocab_size = len(vectorizer.get_vocabulary())

In [8]:
x_train = vectorizer(np.array(sentences)).numpy()

In [11]:
y_train = tf.keras.utils.pad_sequences(labels,maxlen=max_len, padding='post',value=0)

In [12]:
print("Shape of X Train: ", x_train.shape)
print("Shape of Y Train: ", y_train.shape)

Shape of X Train:  (6, 6)
Shape of Y Train:  (6, 6)


In [13]:
num_classes = 4 # 0: PAD, 1: O, 2: PER, 3: ORG

In [16]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(max_len,)),

    tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=12, mask_zero=True),

    
    # Bidirectional with return_sequences=True to retain all 6 timesteps

    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(
        units=16, return_sequences=True
    )),

    tf.keras.layers.TimeDistributed(tf.keras.layers.Dense(
        units=num_classes, activation='softmax'
    ))
])

In [20]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss='sparse_categorical_crossentropy',
    metrics = ['accuracy']
        
)

In [21]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)              │ (None, 6, 12)               │             408 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional_1 (Bidirectional)      │ (None, 6, 32)               │           3,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed (TimeDistributed)   │ (None, 6, 4)                │             132 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,252 (16.61 KB)

 Trainable params: 4,252 (16.61 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
history = model.fit(
    x_train,
    y_train,
    epochs=70,
    verbose=1
)

Epoch 1/70
1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step - accuracy: 0.2000 - loss: 1.3883
Epoch 2/70
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.4857 - loss: 1.3628
Epoch 3/70
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.4857 - loss: 1.3356
Epoch 4/70
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.4857 - loss: 1.3039
Epoch 5/70
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.4857 - loss: 1.2663
Epoch 6/70
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.4857 - loss: 1.2223
Epoch 7/70
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step - accuracy: 0.4857 - loss: 1.1722
Epoch 8/70
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - accuracy: 0.4857 - loss: 1.1178
Epoch 9/70
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.4857 - loss: 1.0624
Epoch 10/70
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 93ms/step - accuracy: 0.4857 - loss: 1.0093
Epoch 11/70
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.4857 - loss: 0.9590
Epoch 12/70
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.6286 - loss: 0.9077
E

In [23]:
# Test on a new sequence containing ambiguous company/name positions
test_sentence = np.array(["elon visited apple yesterday"])
test_vec = vectorizer(test_sentence).numpy()

preds = model.predict(test_vec)  # shape: (1, 6, 4)
predicted_classes = np.argmax(preds[0], axis=-1)

tag_map = {0: "PAD", 1: "O", 2: "PER", 3: "ORG"}
tokens = test_sentence[0].split()

for i in range(len(tokens)):
    print(f"Word: {tokens[i]:<10} -> Tag: {tag_map[predicted_classes[i]]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 595ms/step
Word: elon       -> Tag: PER
Word: visited    -> Tag: O
Word: apple      -> Tag: ORG
Word: yesterday  -> Tag: O
